# Day 1 · 技术版图与问题定义

**配套讲义**: [`days/day-01.md`](../days/day-01.md) ｜ **本地能跑一部分，训练/推理要 GPU**

建立一张属于自己的「多模态技术树」：说清三条技术路线（对比学习式 / 融合式 / 外挂式）各自把图塞进了哪一层，并在云上让 Qwen2.5-VL 真的描述一张商品图。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w1.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 0. 环境自检

先确认自己在哪个环境跑（本地还是云上），两边的期望是不一样的。

In [ ]:
import platform, sys, shutil

print("Python :", sys.version.split()[0])
print("系统   :", platform.platform())

def has(mod):
    try:
        m = __import__(mod)
        return getattr(m, "__version__", "已安装")
    except ImportError:
        return None

for pkg in ("torch", "transformers", "PIL", "numpy"):
    v = has(pkg)
    print(f"{pkg:14s} {'✗ 缺失' if v is None else v}")

try:
    import torch
    print("CUDA 可用:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("设备    :", torch.cuda.get_device_name(0))
        print("显存    :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
except ImportError:
    print("\n（本地没装 torch 是正常的 —— 本地只负责读文档改代码）")

print("\n磁盘可用:", round(shutil.disk_usage(".").free / 1024**3, 1), "GB")

## 1. 模型下载状态

**先看要下多少，再决定下不下。** 云机器系统盘经常只有 30–50 GB，
下完模型就满了 —— 这是新手最常踩的坑。

In [ ]:
import subprocess, sys, os
root = os.environ.get("MODEL_ROOT", "/root/autodl-tmp/models")
r = subprocess.run([sys.executable, "../scripts/download_model.py",
                    "--status", "--root", root],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

## 2. 第一次真实推理

现在让 Qwen2.5-VL 描述一张商品图。

⚠️ 这一步需要**云 GPU + 已下载的模型权重**。本地跑会直接报错，那是预期的 ——
把这段代码留到云上跑。

**关键**：跑完把模型的原始回答抄进 `progress/daily-log.md`。8 周后你要拿它做对比。

In [ ]:
# ⚠️ 需要云 GPU + 已下载权重。本地会报错，跳过即可。
import os
MODEL = os.environ.get("MODEL_PATH", "/root/autodl-tmp/models/Qwen2.5-VL-3B-Instruct")

try:
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    from PIL import Image

    processor = AutoProcessor.from_pretrained(MODEL)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL, torch_dtype="auto", device_map="auto").eval()

    # 有真实商品图就用真实的；没有就先用一张纯色图看流程通不通
    img_path = "../assets/sample_product.jpg"
    if not os.path.exists(img_path):
        img = Image.new("RGB", (448, 448), (240, 236, 228))
        print("（没找到示例商品图，用一张纯色图先验证流程）")
    else:
        img = Image.open(img_path).convert("RGB")
        print("用图:", img_path)

    messages = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": "请描述这张图片，然后告诉我它适合什么季节穿。"},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False,
                                         add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    answer = processor.batch_decode(out[:, inputs.input_ids.shape[1]:],
                                    skip_special_tokens=True)[0]
    print("\n=== 模型回答 ===")
    print(answer)
    print("\n→ 把这段原文抄进 progress/daily-log.md，8 周后再回来看")
except ImportError as e:
    print("本地缺少依赖（正常）:", e)
    print("这一步请在云 GPU 机器上跑")
except Exception as e:
    print(f"{type(e).__name__}: {e}")
    print("如果是找不到模型，先跑 Day 1 第三节的下载命令")

## 3. 技术树（手绘）

不用打字，拿张纸画，拍照存 `assets/tech-tree.png`。

至少要画出来：

```
图像 → [?] → [?] → 语言模型 → 文本
        ①      ②          ③
      编码器   连接器     LLM
```

然后在三条路线上各标一个代表：CLIP / Flamingo / LLaVA。

画完问自己：**BLIP-2 的 Q-Former 属于①②③里的哪一环？它和 LLaVA 的 MLP 有什么本质区别？**

## 4. 三行打卡

复制下面这段，填好贴进 `progress/daily-log.md`。

In [ ]:
print("""今日打卡（填好贴进 progress/daily-log.md）
─────────────────────────────────────────
[学到] 三条技术路线的差别是 ______，我原来以为 ______
[产出] assets/tech-tree.png；模型原始回答：______
[卡住] ______（没卡就写「无」）
─────────────────────────────────────────""")

## 验收清单

- [ ] 能**不看资料**说出三条技术路线的差异，以及为什么客服场景选外挂式
- [ ] 云上能加载 Qwen2.5-VL-3B 并让它描述一张商品图（哪怕答案很一般）
- [ ] `assets/` 里有一张自己画的技术树（手绘拍照也算）
- [ ] 知道「哪些事在本地做、哪些必须上云」—— 这一条决定了你 8 周花 ¥300 还是 ¥3000

**卡住了？** 回看 [`days/day-01.md`](../days/day-01.md) 第五节「容易踩的坑」。

> **明天**：`days/day-02.md` —— 手写 ViT：patch embedding + attention，打印每一层的 shape